# Camada Bronze — `notas`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Descrição da tabela:** Lançamentos de notas por disciplina e data  
**Arquivo bruto:** `Atividade11 - notas.txt`

In [1]:
import json
import sys
from pathlib import Path

# Garante que notebooks/ esteja no sys.path independente do diretório de execução
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import (
    get_project_paths,
    ingest_raw_file,
    profile_bronze_table,
    save_bronze_outputs,
)

ROOT_DIR, RAW_DIR, BRONZE_DIR = get_project_paths()
print(f"Origem: {RAW_DIR / 'Atividade11 - notas.txt'}")
print(f"Destino Bronze: {BRONZE_DIR}")

Origem: C:\Projects\trabalho-final-mineacao-de-dados\Bases TRabalho Final\Atividade11 - notas.txt
Destino Bronze: C:\Projects\trabalho-final-mineacao-de-dados\data\bronze


In [2]:
df, meta = ingest_raw_file(
    dataset_name='notas',
    filename='Atividade11 - notas.txt',
    sep='|',
    mixed_sep=False,
    raw_dir=RAW_DIR,
)

profile = profile_bronze_table(df, meta)
parquet_path, profile_path = save_bronze_outputs(df, profile, 'notas', BRONZE_DIR)

print(f"Linhas ingeridas: {meta['rows']:,} | Tamanho bruto: {meta['size_mb']} MB")
print(f"IDs únicos de aluno: {profile['unique_id_aluno']:,} | Duplicatas exatas: {profile['exact_duplicates']:,}")
print(f"Salvo em: {parquet_path.name} e {profile_path.name}")
df.head(5)

Linhas ingeridas: 600,000 | Tamanho bruto: 15.7152 MB
IDs únicos de aluno: 349,170 | Duplicatas exatas: 0
Salvo em: notas.parquet e notas_profile.json


,id_aluno,disciplina,nota,data,_bronze_source_file,_bronze_ingested_at
0,281506,Quim,5.3,2022-09-07,Atividade11 - notas.txt,2026-11-13T18:55:02Z
1,498846,Port,2.49,2024-04-20,Atividade11 - notas.txt,2026-11-13T18:55:02Z
2,209233,Fis,-0.77,2022-07-14,Atividade11 - notas.txt,2026-11-13T18:55:02Z
3,213008,Quim,9.87,2020-09-16,Atividade11 - notas.txt,2026-11-13T18:55:02Z
4,470857,Port,6.29,2021-05-22,Atividade11 - notas.txt,2026-11-13T18:55:02Z


In [3]:
print("=== Campos vazios por coluna ===")
print(json.dumps(profile["empty_counts"], ensure_ascii=False, indent=2))
if profile["anomalies"]:
    print("\n=== Anomalias detectadas para tratamento na Silver ===")
    print(json.dumps(profile["anomalies"], ensure_ascii=False, indent=2))

=== Campos vazios por coluna ===
{
  "id_aluno": 0,
  "disciplina": 0,
  "nota": 0,
  "data": 0
}

=== Anomalias detectadas para tratamento na Silver ===
{
  "nota_menor_0": 85474,
  "nota_maior_10": 85653
}
